<a href="https://colab.research.google.com/github/checkcode01/60-Days-Challenge/blob/main/Day11_HR_Document_Retrieval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install scikit-learn nltk

In [ ]:
import re
import nltk
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

nltk.download('stopwords')

from nltk.corpus import stopwords

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [14]:
knowledge_base = [

      # Leave Policy
          "Employees are entitled to 20 annual leave days",
              "Sick leave can be availed with medical approval",
                  "Work from home is allowed during emergencies",

                      # Performance
                          "Performance reviews are conducted twice a year",
                              "Employees receive feedback during appraisal cycles",
                                  "Managers evaluate employees based on goals",

                                      # Learning
                                          "Employees can access online learning platforms",
                                              "Technical certification programs are reimbursed",
                                                  "Leadership training is available for managers",

                                                      # Compensation
                                                          "Salary revisions occur annually",
                                                              "Employees receive medical insurance benefits",
                                                                  "Bonus payouts depend on company performance",

                                                                      # Work Culture
                                                                          "The company promotes diversity and inclusion",
                                                                              "Employees are encouraged to collaborate",
                                                                                  "Flexible work culture improves employee wellbeing",

                                                                                      # Recruitment
                                                                                          "Internal job postings are shared monthly",
                                                                                              "Employees can refer candidates for hiring",
                                                                                                  "Interview feedback must be submitted within two days",

                                                                                                      # HR Support
                                                                                                          "Employees can contact HR for grievance support",
                                                                                                              "HR conducts onboarding sessions for new hires"
                                                                                                              ]


In [16]:
class PreprocessingModule:

    def __init__(self):
        self.stop_words = set(stopwords.words('english'))

    def transform(self, text):
        if not text.strip():
            return "ERROR_EMPTY"

        if len(text.strip()) == 1:
            return "ERROR_SINGLE_CHARACTER"

        if re.fullmatch(r'[^a-zA-Z]+', text):
            return "ERROR_SYMBOLS_NUMBERS"

        text = text.lower()
        text = re.sub(r'[^a-zA-Z\s]', '', text)

        words = text.split()
        filtered_words = [
            word for word in words
            if word not in self.stop_words
        ]

        return " ".join(filtered_words)


In [17]:
class VectorizerModule:

    def __init__(self):

        self.vectorizer = TfidfVectorizer()

    def fit(self, corpus):

        self.matrix = self.vectorizer.fit_transform(corpus)

    def transform(self, query):

        return self.vectorizer.transform([query])

In [18]:
preprocessor = PreprocessingModule()

cleaned_corpus = [
    preprocessor.transform(doc)
    for doc in knowledge_base
]

In [19]:
vectorizer = VectorizerModule()

vectorizer.fit(cleaned_corpus)

corpus_matrix = vectorizer.matrix

In [21]:
def retrieve(query, corpus_matrix, top_k=3):

    cleaned_query = preprocessor.transform(query)

    # Error handling
    if cleaned_query.startswith("ERROR"):

        print(f"\nInvalid Query: {cleaned_query}")
        return

    # Vectorize query
    query_vector = vectorizer.transform(cleaned_query)

    # Similarity scores
    similarities = cosine_similarity(
        query_vector,
        corpus_matrix
    ).flatten()

    # Relevance threshold
    if similarities.max() < 0.1:

        print("\nNo relevant document found")
        return

    # Rank documents
    ranked_indices = similarities.argsort()[::-1]

    print(f"\nQUERY: {query}\n")

    for idx in ranked_indices[:top_k]:

        print(f"Document: {knowledge_base[idx]}")
        print(f"Score: {similarities[idx]:.3f}")
        print("-" * 50)

In [22]:
retrieve(
    "Can I take sick leave?",
    corpus_matrix
)


QUERY: Can I take sick leave?

Document: Sick leave can be availed with medical approval
Score: 0.624
--------------------------------------------------
Document: Employees are entitled to 20 annual leave days
Score: 0.296
--------------------------------------------------
Document: HR conducts onboarding sessions for new hires
Score: 0.000
--------------------------------------------------


In [23]:
retrieve(
    "Is work from home allowed?",
    corpus_matrix
)


QUERY: Is work from home allowed?

Document: Work from home is allowed during emergencies
Score: 0.857
--------------------------------------------------
Document: Flexible work culture improves employee wellbeing
Score: 0.193
--------------------------------------------------
Document: HR conducts onboarding sessions for new hires
Score: 0.000
--------------------------------------------------


In [28]:
retrieve(
    "Does company provide certifications?",
    corpus_matrix
)


QUERY: Does company provide certifications?

Document: The company promotes diversity and inclusion
Score: 0.453
--------------------------------------------------
Document: Bonus payouts depend on company performance
Score: 0.412
--------------------------------------------------
Document: HR conducts onboarding sessions for new hires
Score: 0.000
--------------------------------------------------


In [25]:
retrieve(
    "How are bonuses calculated?",
    corpus_matrix
)


No relevant document found


In [27]:
retrieve(
    "Can employees refer candidates?",
    corpus_matrix
)


QUERY: Can employees refer candidates?

Document: Employees can refer candidates for hiring
Score: 0.835
--------------------------------------------------
Document: Employees are encouraged to collaborate
Score: 0.132
--------------------------------------------------
Document: Employees receive medical insurance benefits
Score: 0.102
--------------------------------------------------


In [29]:
retrieve(
    "Does company support diversity?",
    corpus_matrix
)


QUERY: Does company support diversity?

Document: The company promotes diversity and inclusion
Score: 0.548
--------------------------------------------------
Document: Employees can contact HR for grievance support
Score: 0.297
--------------------------------------------------
Document: Bonus payouts depend on company performance
Score: 0.218
--------------------------------------------------


In [30]:
retrieve(
    "employee support",
    corpus_matrix
)


QUERY: employee support

Document: Employees can contact HR for grievance support
Score: 0.350
--------------------------------------------------
Document: Flexible work culture improves employee wellbeing
Score: 0.294
--------------------------------------------------
Document: Interview feedback must be submitted within two days
Score: 0.000
--------------------------------------------------


In [31]:
retrieve(
    "career help",
    corpus_matrix
)


No relevant document found


In [32]:
retrieve(
    "best pizza recipe",
    corpus_matrix
)


No relevant document found


In [33]:
retrieve(
      "airplane engine maintenance",
          corpus_matrix
          )
)

SyntaxError: unmatched ')' (815558165.py, line 5)

Retrieval Failure Analysis
Query: "employee support"
Cause: The query is too broad and overlaps multiple HR topics.
Query: "career help"
Cause: TF-IDF relies on exact vocabulary and does not understand semantic synonyms deeply.
Query: "best pizza recipe"
Cause: No overlapping HR vocabulary exists in the knowledge base.
Query: "airplane engine maintenance"
Cause: Query is completely outside the HR domain.

Vocabulary Mismatch Problem in TF-IDF
TF-IDF retrieval depends heavily on exact word overlap.
Example:
"salary" and "compensation"
"learning" and "training"
may represent similar meanings to humans, but TF-IDF treats them as different words if vocabulary does not overlap.
This causes retrieval failures for synonym-based queries.
This limitation explains why modern AI systems use embeddings instead of simple TF-IDF vectors.
Embeddings capture semantic meaning rather than exact keyword matching.
Modern systems like ChatGPT use embeddings to understand:
synonyms
context
intent
semantic relationships
far more effectively than TF-IDF.

User Query
     ↓
     PreprocessingModule
     (cleaning text)
          ↓
          TF-IDF Vectorizer
          (convert text to vectors)
               ↓
               Cosine Similarity
               (compare query vs documents)
                    ↓
                    Ranked Documents
                         ↓
                         Threshold Filter
                         (remove irrelevant results)
                              ↓
                              Final Retrieval Results